In [ ]:
import json
import logging

from openeo.rest.udp import build_process_dict
from utils import udp_params, utils

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

# Parameters

In [ ]:
forest_baseline_mask = udp_params.FOREST_BASELINE_DATACUBE
s1_dB = udp_params.SENTINEL_1_DATACUBE

spatial_extent = udp_params.SPATIAL_EXTENT

resample_spatial_resolution = udp_params.SPATIAL_RESOLUTION

temporal_variability_threshold = udp_params.TEMPORAL_VARIABILITY_THRESHOLD
flattening_threshold = udp_params.FLATTENING_THRESHOLD
logistic_sse_threshold = udp_params.LOGISTIC_SSE_THRESHOLD

min_connected_area = udp_params.MIN_CONNECTED_AREA

In [ ]:
parameters = [
    forest_baseline_mask,
    s1_dB,
    spatial_extent,
    resample_spatial_resolution,
    temporal_variability_threshold,
    flattening_threshold,
    logistic_sse_threshold,
    min_connected_area,
]

# UDP

In [ ]:
# initial spatial filter
# also creates client-side DataCube instances

forest_baseline_mask = connection.datacube_from_process(
    "filter_bbox",
    data=forest_baseline_mask,
    extent=spatial_extent,
)
s1_dB = connection.datacube_from_process(
    "filter_bbox",
    data=s1_dB,
    extent=spatial_extent,
)

In [ ]:
# resample onto S1 grid @ resample_spatial_resolution
forest_baseline_mask = forest_baseline_mask.resample_cube_spatial(
    s1_dB,
    method="near",  # Reference implementation uses rioxarray reproject_match() which has default Resampling.nearest
)

## temporal variability mask

In [ ]:
# mask of interesting pixels
# temporal_variability_mask = s1_dB.band("sd") >= temporal_variability_threshold

# cannot use band math with Parameters 🙁
# https://forum.dataspace.copernicus.eu/t/udp-parameter-not-applied/5282

gte_temporal_variability_threshold = openeo.UDF.from_file(
    "../udf/binary_operator.py",
    runtime="Python",
    version="3.11",
    context={
        "operator": "gte",
        "argument": temporal_variability_threshold,
    },
)

temporal_variability_mask = (
    s1_dB.filter_bands("sd")
    .apply(gte_temporal_variability_threshold)
    .convert_data_type("bool")
    .band("sd")
)

# flattening mask

In [ ]:
# TODO: ATBD says denominator is abs(gamma_max)
# but code uses p05
# Dascalu 2023 has abs(gamma_max)


def flattening_reducer(
    bands: openeo.processes.ProcessBuilder,
) -> openeo.processes.ProcessBuilder:
    p05 = bands.array_element(label="p05")
    p95 = bands.array_element(label="p95")
    return (p95 - p05) / p05.absolute()


flattening = s1_dB.reduce_bands(flattening_reducer)

In [ ]:
assert flattening._in_bandmath_mode()

In [ ]:
# mask of good pixels where flattening >= flattening_threshold
# flattening_mask = flattening >= flattening_threshold

# cannot use band math with Parameters 🙁
# https://forum.dataspace.copernicus.eu/t/udp-parameter-not-applied/5282

gte_flattening_threshold = openeo.UDF.from_file(
    "../udf/binary_operator.py",
    runtime="Python",
    version="3.11",
    context={
        "operator": "gte",
        "argument": flattening_threshold,
    },
)

flattening = flattening.add_dimension("bands", label="flattening", type="bands")
flattening_mask = (
    flattening.apply(gte_temporal_variability_threshold)
    .convert_data_type("bool")
    .band("flattening")
)

# SSE goodness of fit mask

In [ ]:
# sse_threshold_mask = s1_dB.band("min_sse") <= logistic_sse_threshold

# cannot use band math with Parameters 🙁
# https://forum.dataspace.copernicus.eu/t/udp-parameter-not-applied/5282

lte_logistic_sse_threshold = openeo.UDF.from_file(
    "../udf/binary_operator.py",
    runtime="Python",
    version="3.11",
    context={
        "operator": "lte",
        "argument": logistic_sse_threshold,
    },
)

sse_threshold_mask = (
    s1_dB.filter_bands("min_sse")
    .apply(lte_logistic_sse_threshold)
    .convert_data_type("bool")
    .band("min_sse")
)

# Combine all masks

In [ ]:
# forest_baseline_mask.band("B0") & temporal_variability_mask & flattening_mask & sse_threshold_mask
# BandMathException: 'Band math' between bands of different data cubes is not supported yet. 🙁

In [ ]:
# in order to do band math, we need to stack all of the inputs into a single DataCube
# https://github.com/Open-EO/openeo-python-client/issues/748

b1 = forest_baseline_mask.rename_labels(
    dimension="bands", target=["forest_baseline_mask"]
)
b2 = temporal_variability_mask.add_dimension(
    "bands", label="temporal_variability_mask", type="bands"
)
b3 = flattening_mask.add_dimension("bands", label="flattening_mask", type="bands")
b4 = sse_threshold_mask.add_dimension("bands", label="sse_threshold_mask", type="bands")

combined_masks = b1.merge_cubes(b2).merge_cubes(b3).merge_cubes(b4)

In [ ]:
# ⚠️ there are some horrible artifacts in this mask
# https://forum.dataspace.copernicus.eu/t/load-stac-configure-nodata/5267
# in tiles over water

# deforestation event = 1
deforestation_event_mask = (
    combined_masks.band("temporal_variability_mask")
    & combined_masks.band("flattening_mask")
    & combined_masks.band("forest_baseline_mask")
    & combined_masks.band("sse_threshold_mask")
)

In [ ]:
# ⚠️ there are some horrible artifacts in this mask
# https://forum.dataspace.copernicus.eu/t/load-stac-configure-nodata/5267
# in tiles over water

inverse_deforestation_event_mask = utils.logical_not(deforestation_event_mask)

# Apply deforestation event mask to `min_sse_t`

In [ ]:
min_sse_t_masked = s1_dB.band("min_sse_t").mask(inverse_deforestation_event_mask)

## mask based on connectivity

of the natural forest remaining, are regions of forest too small to meet the minimum connected area threshold?

In [ ]:
remaining_forest_mask = (
    combined_masks.band("forest_baseline_mask") & inverse_deforestation_event_mask
)

In [ ]:
connectivity_udf = openeo.UDF.from_file(
    "../udf/connectivity_mask.py",
    runtime="Python",
    version="3.11",
    context={
        # "pixel_area": resample_spatial_resolution * resample_spatial_resolution,
        "pixel_area": openeo.processes.multiply(
            resample_spatial_resolution,
            resample_spatial_resolution,
        ),
        "min_connected_area": min_connected_area,
    },
)

In [ ]:
# mask where 1 = small region to be excluded
small_region_mask = remaining_forest_mask.apply_neighborhood(
    connectivity_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    # overlap needs to be big enough the reasonably allow for min_pixels
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

In [ ]:
# apply_neighborhood UDF seems to return float32, even if it's a mask
# data types: https://github.com/locationtech/geotrellis/blob/master/raster/src/main/scala/geotrellis/raster/CellType.scala
small_region_mask = small_region_mask.convert_data_type("bool")

In [ ]:
# stack min_sse_t_masked + small_region_mask into a single datacube
# in preparation for the nearest neighbour fill UDF

_data = min_sse_t_masked.add_dimension("bands", label="data", type="bands")
_mask = small_region_mask.add_dimension("bands", label="mask", type="bands")
combined = _data.merge_cubes(_mask)

In [ ]:
nearest_neighbour_fill_udf = openeo.UDF.from_file(
    "../udf/nearest_neighbour_fill.py",
    runtime="Python",
    version="3.11",
)

In [ ]:
min_sse_t = combined.apply_neighborhood(
    nearest_neighbour_fill_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

# Serialise UDP

In [ ]:
summary = "Detect decimal year of deforestation"
description = "// TODO"

udp_spec = build_process_dict(
    min_sse_t,
    process_id="deforestation",
    summary=summary,
    description=description,
    parameters=parameters,
    returns={
        "description": "A DataCube with dtype float",
        "schema": {"type": "object", "subtype": "datacube"},
    },
)

In [ ]:
with open("udp.json", "w") as f:
    json.dump(udp_spec, f, indent=2)